# Week 3 - Data Wrangling

In this notebook we will practice data cleaning and wrangling using a *messy* version of the Titanic dataset.

Topics covered:
- Inspecting data
- Preprocessing
    - Categorical x Numerical Data
    - Fixing column types
    - Standardizing Categorical Values
    - Missing values (Identifying and Imputation)
- Feature Engineering
- Exporting and Loading Cleaned Data

### SETUP

In [1]:
# Uncomment if needed.
# %pip install pandas numpy

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

def locate_data(filename):
    candidates = [Path(filename), Path('Week_3') / filename,
                  Path('raw') / filename, Path('../raw') / filename]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f'Upload {filename} to Colab or keep it in the supplied dataset folder.')

data_path = locate_data('messy_titanic.csv')

## Dataset

This is a modified Titanic teaching dataset with deliberately inconsistent formatting, artificial passenger identifiers and substituted names. It is not identical to the Week 2 CSV. The `USD` suffix is a formatting defect to remove, not evidence of a currency conversion.

| Column | Meaning |
| --- | --- |
| `passengerId` | Artificial passenger identifier |
| `survived` | Outcome: 0 = no, 1 = yes |
| `pclass` | Ticket class: 1, 2, 3 (ordered categories) |
| `name` | Substituted teaching name |
| `sex` | Sex label as supplied |
| `age` | Age in years; fractional ages are meaningful |
| `sibsp`, `parch` | Counts of siblings/spouses and parents/children aboard |
| `fare` | Recorded fare, sometimes containing an artificial text suffix |
| `adult_male` | Supplied Boolean attribute; not recomputed after imputation |
| `deck` | Deck letter; the input misspells this as `dreck` |
| `embark_town` | Embarkation town |

## 1. Inspecting the dataset
### Load the CSV once with the default separator. What goes wrong?

In [3]:
preview = pd.read_csv(data_path)
print(preview.shape)
print(preview.head(2))
# The default comma separator leaves the semicolon-delimited data in one column.

(891, 1)
  ;survived;pclass;sex;age;sibsp;parch;fare;adult_male;dreck;embark_town;passengerId;name
0  0;0;3;Male;22.0;1;0;7.25;True;;Southampton;T3-...                                     
1  1;1;1;female;38.0;1;0;71.2833;False;C;Cherbour...                                     


Inspect the first three text lines using `data_path.read_text().splitlines()[:3]`. This works on Windows and Colab. Identify the delimiter before trying again.

In [4]:
print('\n'.join(data_path.read_text(encoding='utf-8').splitlines()[:3]))

;survived;pclass;sex;age;sibsp;parch;fare;adult_male;dreck;embark_town;passengerId;name
0;0;3;Male;22.0;1;0;7.25;True;;Southampton;T3-1912-S-P0001;Dustin Flores
1;1;1;female;38.0;1;0;71.2833;False;C;Cherbourg;T1-1912-C-P0001;Mary Tyler


Read the file with the correct `sep` into `raw`. Preserve `raw` unchanged throughout the notebook.

In [5]:
raw = pd.read_csv(data_path, sep=';')
print(raw.head())

   Unnamed: 0  survived  pclass     sex   age  sibsp  parch     fare  \
0           0         0       3    Male  22.0      1      0     7.25   
1           1         1       1  female  38.0      1      0  71.2833   
2           2         1       3  female  26.0      0      0    7.925   
3           3         1       1  female  35.0      1      0     53.1   
4           4         0       3    Male  35.0      0      0     8.05   

   adult_male dreck  embark_town      passengerId            name  
0        True   NaN  Southampton  T3-1912-S-P0001   Dustin Flores  
1       False     C    Cherbourg  T1-1912-C-P0001      Mary Tyler  
2       False   NaN  Southampton  T3-1912-S-P0002    Anna Mendoza  
3       False     C  Southampton  T1-1912-S-P0001   Debra Pearson  
4        True   NaN  Southampton  T3-1912-S-P0003  Anthony Taylor  


### Q 1.1. Remove the saved index
Create `df` by dropping `Unnamed: 0` from `raw` and taking a copy. Explain why a DataFrame index is different from `passengerId`.

In [6]:
df = raw.drop(columns=['Unnamed: 0']).copy()
print(df.head())

   survived  pclass     sex   age  sibsp  parch     fare  adult_male dreck  \
0         0       3    Male  22.0      1      0     7.25        True   NaN   
1         1       1  female  38.0      1      0  71.2833       False     C   
2         1       3  female  26.0      0      0    7.925       False   NaN   
3         1       1  female  35.0      1      0     53.1       False     C   
4         0       3    Male  35.0      0      0     8.05        True   NaN   

   embark_town      passengerId            name  
0  Southampton  T3-1912-S-P0001   Dustin Flores  
1    Cherbourg  T1-1912-C-P0001      Mary Tyler  
2  Southampton  T3-1912-S-P0002    Anna Mendoza  
3  Southampton  T1-1912-S-P0001   Debra Pearson  
4  Southampton  T3-1912-S-P0003  Anthony Taylor  


### Q 1.2. How many rows and columns are there?
Checkpoint after removing the saved index: 891 rows and 12 columns.

In [7]:
print(df.shape)
assert df.shape == (891, 12)

(891, 12)


### Q.1.3 Check of each column data type

- Is every column data type the most appropriate for it?
- Are there any column with null/missing values?

In [8]:
df.info()
print(df.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    object 
 3   age          714 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null    int64  
 6   fare         846 non-null    object 
 7   adult_male   891 non-null    bool   
 8   dreck        203 non-null    object 
 9   embark_town  889 non-null    object 
 10  passengerId  891 non-null    object 
 11  name         891 non-null    object 
dtypes: bool(1), float64(1), int64(4), object(6)
memory usage: 77.6+ KB
survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare            45
adult_male       0
dreck          688
embark_town      2
passengerId      0
name             0
dtype: 

### Q.1.4 Verify descritive statistics
- All numeric columns are being displayed?
- Those descritive statistcs are meaningfull for every column?

In [9]:
print(df.describe())
# IDs and category codes should not be interpreted as measured quantities.
# Fare is absent from numeric summaries until its text formatting is repaired.

         survived      pclass         age       sibsp       parch
count  891.000000  891.000000  714.000000  891.000000  891.000000
mean     0.383838    2.308642   29.699118    0.523008    0.381594
std      0.486592    0.836071   14.526497    1.102743    0.806057
min      0.000000    1.000000    0.420000    0.000000    0.000000
25%      0.000000    2.000000   20.125000    0.000000    0.000000
50%      0.000000    3.000000   28.000000    0.000000    0.000000
75%      1.000000    3.000000   38.000000    1.000000    0.000000
max      1.000000    3.000000   80.000000    8.000000    6.000000


## 2. Preprocessing
### Q 2.1. Correct the column name
Rename `dreck` to `deck` in `df`. For example, `df = df.rename(columns={'old_name': 'new_name'})`.

In [10]:
df = df.rename(columns={'dreck': 'deck'})
assert 'deck' in df and 'dreck' not in df

<details>
<summary style="background-color: #fff3cd; padding: 10px; border-radius: 5px; cursor: pointer;">
    <h2 style="display: inline;">
        Understanding Numerical vs. Categorical Data
        <span style="color: red; font-size: 0.8em;">(click to expand/collapse)</span>
    </h2>
</summary>

It is important to understand **what type of data each column represents**.  
Choosing the **correct type** improves **memory usage**, **speeds up operations**, and makes **analysis more reliable**.

---

### **1. Numerical Data**
>
>These are values that represent quantities and support mathematical operations.
>
>**Examples:**
>- Age (`22`, `45.0`)
>- Fare (`71.283`, `15.50`)
>- Number of siblings (`3`, `0`)
>- Any measured or counted quantity
>
>**Two main subtypes:**
>- **int** → whole numbers (0, 1, 2…)
>- **float** → decimal numbers (0.0, 3.14…)
>
>**Numerical columns allow:**
>- statistics (mean, median, std)
>- plotting histograms
>- mathematical operations
>- machine learning models
---
### **2. Categorical Data**
>
>These represent **labels**, **groups**, or **categories**, not quantities.
>
>**Examples:**
>- Sex (`"male"`, `"female"`)
>- Embarkation port (`"C"`, `"Q"`, `"S"`)
>- Passenger class (`"First"`, `"Second"`, `"Third"`)
>- Deck (`"A"`, `"B"`, `"Unknown"`)
>
>**Types commonly used:**
>- **string (object)** → free text labels  
>- **category** → optimized categorical labels (recommended)
>
>**Categorical columns allow:**
>- grouping (`df.groupby("sex")`)
>- value counts
>- filtering
>- faster processing when stored as `category`
---
### **Why this distinction matters**
>
>If a column is stored with the wrong dtype:
>
>- A numeric column stored as string **cannot be computed**:  
>  `"45" + "10"` → `"4510"` (string concatenation!)
>- Numeric category codes need explicit interpretation; arithmetic on nominal codes is generally meaningless
>- Machine learning pipelines **fail or behave poorly**
>- Missing values become difficult to detect and fix
>- Memory usage increases (especially with text)
>
>**So the first step in Data Wrangling is:**
>
>👉 **Identify** whether each column is numerical or categorical  
>👉 **Fix** the dtype accordingly
</details>

## Fixing Column Types

Datasets often contain columns stored with the wrong data type.  
This usually happens when values are mixed (e.g., numbers + text), when the file was exported incorrectly, or when data is collected from multiple sources.

Working with incorrect types can break calculations, comparisons, plots, or machine-learning pipelines.  
So the first step is to **inspect** and **fix** them.

**Common strategies for fixing the column types are:**

___
>**Direct convertion:**
>
>If you know that the column will be able to be directly converted
>```python
>df['column_name'] = df['column_name'].astype(dtype)
>```
___
**1. Convert a column to numeric**
>
>Use when the column **should be a number**, but contains strings, symbols, or inconsistent formatting.
>
>```python
>df["column_name"] = pd.to_numeric(df["column_name"], errors="coerce")
>```
>`errors="coerce"` converts invalid values into **NaN**, which can later be imputed.
___

**2. Clean and convert a column that contains units or extra text**
>
>Useful for columns like "100 USD", "45 years", "200 km".
>
>```python
>df["column_name"] = (
>    df["column_name"]
>        .astype("string")
>        .str.replace(" USD", "", regex=False)
>)
>
># now casting to numeric dtype
>df["column_name"] = pd.to_numeric(df["column_name"], errors="coerce")
>```
___
**3. Convert a column to string (categorical text)**
>
>Use when the column should represent labels, not numbers.
>
>```python
>df["column_name"] = df["column_name"].astype("string")
>```
___
**4. Convert a column to category (optional optimization)**
>
>Useful for columns with repeated labels (e.g., "male", "female", "unknown").
>
>```python
>df["column_name"] = df["column_name"].astype("category")
>```
___
**5. Convert a column to datetime**
>
>For dates like "2020-01-05" or "05/01/2020".
>```python
>df["column_name"] = pd.to_datetime(df["column_name"], errors="coerce")
>```
___
**After fixing all types, it is good practice to check the result:**
>
>- `.info()`
>- `.dtypes`

### Q 2.2 Inspect each column, decide the most appropriate `dtype` and `fix` it:

For this file: remove the literal ` USD` suffix from fare before numeric conversion; distinguish original missingness from parse failures. Keep age as floating point to preserve fractional years. Use integer counts for `sibsp`/`parch`, categorical `pclass`, and validate 0/1 before converting `survived` to Boolean. Do not invent values for unknown outcomes.

In [11]:
assert df['survived'].isin([0, 1]).all()
df['survived'] = df['survived'].astype(bool)

In [12]:
fare_text = df['fare'].astype('string').str.strip()
fare_parsed = pd.to_numeric(fare_text.str.replace(' USD', '', regex=False), errors='coerce')
parse_failures = fare_text.notna() & fare_parsed.isna()
print('Original fare missing:', fare_text.isna().sum())
print('Parse failures after removing suffix:', parse_failures.sum())
df['fare'] = fare_parsed.astype(float)

Original fare missing: 45
Parse failures after removing suffix: 0


In [13]:
df['age'] = pd.to_numeric(df['age'], errors='coerce').astype(float)
print(df.loc[df['age'].notna() & df['age'].mod(1).ne(0), 'age'].head())
# Fractional years are valid: do not truncate them.

57     28.50
78      0.83
111    14.50
116    70.50
122    32.50
Name: age, dtype: float64


In [14]:
for col in ['sibsp', 'parch']:
    assert df[col].ge(0).all() and df[col].mod(1).eq(0).all()
    df[col] = df[col].astype('Int64')

In [15]:
assert df['pclass'].isin([1, 2, 3]).all()
df['pclass'] = pd.Categorical(df['pclass'], categories=[1, 2, 3], ordered=True)

In [16]:
df['passengerId'] = df['passengerId'].astype('string')
df['name'] = df['name'].astype('string')
assert df['passengerId'].notna().all() and df['passengerId'].is_unique

In [17]:
assert df['age'].dropna().ge(0).all()
assert df['fare'].dropna().ge(0).all()
print(df.dtypes)

survived                 bool
pclass               category
sex                    object
age                   float64
sibsp                   Int64
parch                   Int64
fare                  float64
adult_male               bool
deck                   object
embark_town            object
passengerId    string[python]
name           string[python]
dtype: object


## Standardizing Categorical Values

Sometimes you may also find inconsistent or incorrect labels in categorical columns.  
For example, the same category may appear in multiple formats:

- "female", "Female", "F", "f"
- "male", "Male", "M", "m"

These inconsistencies make analysis, filtering, and grouping unreliable.  
To fix this, we **standardize** all categorical values into a consistent format.

---
**1. Convert everything to lowercase**
>
>This removes differences like "Female" vs "female".
>
>```python
>df["column_name"] = df["column_name"].astype("string").str.lower()
>```
---
**2. If there are many variations, a `mapping dictionary` might help:**
>
>```python
>mapping = {
>    "female": "female",
>    "f": "female",
>    "woman": "female",
>    "male": "male",
>    "m": "male",
>    "man": "male"
>}
>
>df["column_name"] = df["column_name"].replace(mapping)
>```
___
**Tips:**
>
>- Verify the frequencies of `categorical columns`
>```python
>df["column_name"].value_counts()
>```
>- Converting to `category` offers optimization in terms of memory footprint, comparinson speed and interpretability
>```python
>df["column_name"] = df["column_name"].astype("category")
>```

### Q.2.3 Verify if any categorical column needs to be standardized
- Inspect columns of dtype `category`, `object`, `bool`, `int`
- Inspect using `df['column'].value_counts()`

Normalize `sex` and `embark_town` with `str.strip().str.lower()`. Use `string` during cleaning to preserve missingness. Check observed labels before converting to `category`; do not silently turn unexpected labels into missing values.

In [18]:
for col in ['sex', 'embark_town']:
    df[col] = df[col].astype('string').str.strip().str.lower()
    print(col, df[col].value_counts(dropna=False))
assert df['sex'].dropna().isin(['male', 'female']).all()
assert df['embark_town'].dropna().isin(['southampton', 'cherbourg', 'queenstown']).all()
df['sex'] = df['sex'].astype('category')

sex sex
male      577
female    314
Name: count, dtype: Int64
embark_town embark_town
southampton    644
cherbourg      168
queenstown      77
<NA>             2
Name: count, dtype: Int64


## Missing values

Most real-world datasets contain missing or inconsistent values.

We must decide how to handle them.

**The main strategies are:**
___
**1. Remove rows with missing values**

>Useful when only a few rows contain missing values.
>
>- Removing rows can change the population represented, even if few rows are removed. Justify the decision.
>
>```python
>df_clean = df.dropna()
>```
___
**2. Remove columns with too many missing values**

>Useful when a column is mostly empty (e.g., >80% missing maybe even lower threshold)
>
>- The column is not essential for the analysis
>
>```python
>df_clean = df.drop(columns=["column_name"])
>```
___
**3. Impute missing values (numeric columns)**

>Common inputs:
>
>- Mean
>
>- Median (more robust to outliers)
>
>- Constant value
>
>```python
># mean
>df["column_name"] = df["column_name"].fillna(df["column_name"].mean())
>
># median
>df["column_name"] = df["column_name"].fillna(df["column_name"].median())
>
># constant value
>df["column_name"] = df["column_name"].fillna(1)
>```
___
**4. Impute missing values (categorical columns)**

>Typical strategies:
>
>Replace with the most frequent category (mode)
>
>Replace with a special label such as "Unknown"
>
>
>```python
># mode
>df["column_name"] = df["column_name"].fillna(df["column_name"].mode()[0])
>
># special label
>df["column_name"] = df["column_name"].fillna("Unknown")
>```
___
**There are more advanced methods of imputation but those above cover most of the problems that you might have initially.**

### Q.2.4 Check for missing values in every column:
- Try to obtain a sum of missing values in each column

In [19]:
print(df.isna().sum())

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare            45
adult_male       0
deck           688
embark_town      2
passengerId      0
name             0
dtype: int64


### Q.2.5 Which column is the most problematic in terms of missing values?

In [20]:
most_missing = df.isna().sum().idxmax()
print(most_missing)

deck


### Q.2.6 Let's find out the percentage of missing values in the most problematic column.

In [21]:
print(df[most_missing].isna().mean() * 100)

77.21661054994388


### Q.2.7 Define a function that will check for missing values in percentage each column that you provide as a param.

**Example:**

```python
def percentage_missing_values(df: pd.DataFrame, col: str) -> float:

    # your code here
    
    return percentage_of_missing
```


In [22]:
def percentage_missing_values(table, col):
    if table.empty:
        raise ValueError('Missing percentage is undefined for an empty table')
    return float(table[col].isna().mean() * 100)
print(percentage_missing_values(df, 'deck'))

77.21661054994388


### Q 2.8. Apply an explicit missing-value policy
For this descriptive exercise, preserve every passenger and create `age_imputed` and `fare_imputed` flags before filling age and fare with their respective medians. Fail clearly if a median cannot be estimated. Keep fractional ages. Fill missing `deck` and `embark_town` with `unknown`, then convert these columns to `category`.

A median is an estimate, not the original value. High missingness alone does not prove a column is useless. Compare group sizes before adopting group-specific medians. For future predictive work, estimate imputation and scaling statistics from training data only.

Checkpoint before filling: age 177 missing, fare 45, deck 688, embarkation town 2.

In [23]:
before_imputation = df.copy(deep=True)
print(before_imputation.isna().sum())

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare            45
adult_male       0
deck           688
embark_town      2
passengerId      0
name             0
dtype: int64


In [24]:
fill_values = {}
for col in ['age', 'fare']:
    df[col + '_imputed'] = df[col].isna()
    median = df[col].median()
    if pd.isna(median):
        raise ValueError(f'No observed values for {col}')
    fill_values[col] = float(median)
    df[col] = df[col].fillna(median)
print(fill_values)

{'age': 28.0, 'fare': 14.4542}


In [25]:
for col in ['deck', 'embark_town']:
    df[col] = df[col].astype('string').fillna('unknown').astype('category')

In [26]:
print(df[['age_imputed', 'fare_imputed']].sum())
assert df['age_imputed'].sum() == 177
assert df['fare_imputed'].sum() == 45

age_imputed     177
fare_imputed     45
dtype: int64


In [27]:
assert df[['age', 'fare', 'deck', 'embark_town']].notna().all().all()
assert len(df) == len(raw)

In [28]:
observed_ages = before_imputation['age'].dropna()
assert df.loc[observed_ages.index, 'age'].eq(observed_ages).all()

In [29]:
print(before_imputation.groupby('pclass', observed=True)['age'].agg(['count', 'median']))
# Small groups can give unstable estimates. This exercise uses the overall median.

        count  median
pclass               
1         186    37.0
2         173    29.0
3         355    24.0


## 3. Feature Engineering

Now that the dataset is clean, with corrected data types, standardized categorical values, and missing values imputed, we can begin improving the dataset by **creating new features**.

Feature Engineering helps models extract more meaningful patterns by transforming the raw data into something more informative and more useful for analysis or machine learning.

---
>**1.Mathematical combinations**
>Create new features by combining existing numeric features.
>
>**Example: family size**
>```python
>df["family_size"] = df["sibsp"] + df["parch"] + 1
>```
---
>**2. Binarizing or grouping categories**
>Transform categories into simpler or grouped labels.
>
>**Example: traveling alone vs not**
>```python
>df["is_alone"] = (df["family_size"] == 1)
>```
---
>**3. Extracting information from existing columns**
>
>**Example: title extraction for a different dataset with names such as `Smith, Mr. John`**
>
>This file uses teaching names without titles; do not apply this pattern to it.
>```python
>df["title"] = df["name"].str.extract(r",\s*([^\.]*)\.", expand=False)
>df["title"] = df["title"].str.strip().str.lower()
>```
---
>**4. Discretization (binning)**
>Convert continuous variables into intervals.
>
>**Example using `cut`:**
>```python
>df["age_group"] = pd.cut(
>    df["age"],
>    bins=[0, 12, 18, 35, 60, 100],
>    labels=["child", "teen", "young_adult", "adult", "senior"]
>)
>```
>**-  `pd.cut` will consider intervals as `( ]`**
>**- `open on left (exclusive)` and `closed on right (inclusive)`**
>
>**Example using `qcut` (data will be split in equal parts)**
>```python
>df["age_group"] = pd.qcut(
>    df["age"],
>    q=5,
>    labels=["child", "teen", "young_adult", "adult", "senior"]
>)
>```
---
>**5. Encoding boolean or numeric signals**
>
>**Example: indicator for high fare**
>```python
>df["high_fare"] = df["fare"] > df["fare"].median()
>```
---
>**6. Interaction features**
>Features created by combining multiple existing ones.
>
>**Example: fare divided by recorded family party size**
>
>This ratio is a constructed feature, not a verified per-person ticket price.
>```python
>df["fare_per_person"] = df["fare"] / df["family_size"]
>```
---
**Example Creating feature set**
>
>```python
>df["family_size"] = df["sibsp"] + df["parch"] + 1
>df["is_alone"] = (df["family_size"] == 1)
>
>df["title"] = df["name"].str.extract(r",\s*([^\.]*)\.", expand=False)
>df["title"] = df["title"].str.strip().str.lower()
>
>df["age_group"] = pd.cut(
>    df["age"],
>    bins=[0, 12, 18, 35, 60, 100],
>    labels=["child", "teen", "young_adult", "adult", "senior"]
>)
>
>df["high_fare"] = df["fare"] > df["fare"].median()
>df["fare_per_person"] = df["fare"] / df["family_size"]
>
>df.head()
>```
---
### Q 3.1. Create a new column called `is_child` that is `True` when **age < 12**.

In [30]:
df['is_child'] = df['age'] < 12
# For imputed ages, this derived label is also uncertain.

### **Q 3.2.** Create a column `family_size` based on `sibsp` and `parch`.

In [31]:
df['family_size'] = df['sibsp'] + df['parch'] + 1
assert df['family_size'].ge(1).all()

### Q 3.3. Create a `class` column based on the `Ticket Class`.
- **1 -> First**
- **2 -> Second**
- **3 -> Third**

- **Set the `dtype` of this column to `category`.**

In [32]:
df['class'] = df['pclass'].map({1: 'First', 2: 'Second', 3: 'Third'}).astype('category')

### Q.3.4. Create a `embarked` column based on the Port where the passenger embarked.
- **Use the initials as the category value.**
- **Set the `dtype` of this column to `category`.**
Map Southampton/Cherbourg/Queenstown to S/C/Q after normalization; preserve unknown as `unknown`.

In [33]:
df['embarked'] = df['embark_town'].astype('string').map({'southampton': 'S', 'cherbourg': 'C', 'queenstown': 'Q', 'unknown': 'unknown'}).astype('category')
assert df['embarked'].notna().all()

### Q 3.5. Create a `fare_bin` column by splitting fare into 4 quantiles using `pd.qcut`.
- use `labels` as a list of `'Low', 'Medium', 'High', 'Very High'`
Quantile boundaries can coincide when values repeat; inspect boundaries and do not assume four distinct bins exist in every dataset.

In [34]:
df['fare_bin'], fare_edges = pd.qcut(df['fare'], q=4, labels=['Low', 'Medium', 'High', 'Very High'], retbins=True)
print(fare_edges)
print(df['fare_bin'].value_counts(sort=False))

[  0.       7.925   14.4542  29.85   512.3292]
fare_bin
Low          231
Medium       242
High         195
Very High    223
Name: count, dtype: int64


### Q 3.6. Create a column `age_fare_ratio` equal to `age / fare`, and comment on whether this feature makes sense.
- be careful about zero-division
Represent ratios with zero fare as missing; explain the units and why this feature may have little practical meaning.

In [35]:
df['age_fare_ratio'] = df['age'] / df['fare'].where(df['fare'].ne(0))
assert np.isfinite(df['age_fare_ratio'].dropna()).all()
print('Undefined zero-fare ratios:', df['age_fare_ratio'].isna().sum())
# Years per fare unit has no clear causal interpretation; do not assume usefulness.

Undefined zero-fare ratios: 14


### Q 3.7. Create `age_group` **(you decide the bins and labels)** and count how many passengers exist in each `age_group`.

In [36]:
df['age_group'] = pd.cut(df['age'], bins=[0, 12, 18, 35, 60, np.inf], right=False, labels=['child', 'teen', 'young_adult', 'adult', 'senior'])
print(df['age_group'].value_counts(sort=False))
# Left-closed intervals make age 12 a teen and keep is_child consistent.

age_group
child           68
teen            45
young_adult    543
adult          209
senior          26
Name: count, dtype: int64


### Q 3.8. Extract a surname
For these substituted teaching names, take the last whitespace-separated token. Explain why this is only a dataset-specific convention and not a general rule for human names.

In [37]:
df['surname'] = df['name'].str.split().str[-1]
print(df[['name', 'surname']].head())

             name  surname
0   Dustin Flores   Flores
1      Mary Tyler    Tyler
2    Anna Mendoza  Mendoza
3   Debra Pearson  Pearson
4  Anthony Taylor   Taylor


### **Q 3.9.** Using the engineered features, compute a summary showing:
- average `fare` per `age_group`
- average `family_size` per `class`

In [38]:
print(df.groupby('age_group', observed=True)['fare'].agg(['count', 'mean']))
print(df.groupby('class', observed=True)['family_size'].agg(['count', 'mean']))

             count       mean
age_group                    
child           68  30.777207
teen            45  26.078340
young_adult    543  24.836456
adult          209  47.281840
senior          26  33.908496


        count      mean
class                  
First     216  1.773148
Second    184  1.782609
Third     491  2.008147


## 4. Export Dataset

Now that our dataset is clean, we might want to save time by not needing to preprocess it again.

Exporting it is a good way to avoid running all cleaning steps every time you start your notebook.

There are many formats you can choose, and the most common are:

---

### ✔️ Export to CSV
>```python
>df_clean.to_csv("titanic_clean.csv", index=False)
>```
>
>- `index=False` avoids saving the DataFrame index as an extra column.
>- CSV files are human-readable and compatible with most tools.
---
### ✔️ Export to Excel
>```python
>df_clean.to_excel("titanic_clean.xlsx", index=False)
>```
>
>- Requires `openpyxl` or `xlsxwriter` installed.
>- Useful when sharing data with non-programmers.
---
### ✔️ Export to JSON
>```python
>df_clean.to_json("titanic_clean.json", orient="records")
>```
>
>**Common orientations:**
>
>- `"records"` → list of dictionaries (most common)
>- `"split"` → dictionary with `index`, `columns`, `data`
>- `"table"` → includes metadata, useful for interoperability
---
### ✔️ Loading Back the Dataset Later
>```python
>df_loaded = pd.read_csv("titanic_clean.csv")
>```
>
>**or:**
>
>```python
>df_loaded = pd.read_json("titanic_clean.json")
>```
---
**if you are curious, search for `pickle` and `parquet`.**

Keep the preparation code and source data so results remain reproducible. CSV does not preserve categorical dtypes; restore them explicitly when needed.

### Q 4.1. Export the dataset to `csv`. Set `index=True` as an additional param.

Write all exercise exports under `outputs/`; never overwrite the supplied CSV.

In [39]:
Path('outputs').mkdir(exist_ok=True)
df.to_csv('outputs/titanic_with_index.csv', index=True)

### Q 4.2 Load into a variable named `df_loaded`

In [40]:
df_loaded = pd.read_csv('outputs/titanic_with_index.csv')
print(df_loaded.columns)
assert 'Unnamed: 0' in df_loaded

Index(['Unnamed: 0', 'survived', 'pclass', 'sex', 'age', 'sibsp', 'parch',
       'fare', 'adult_male', 'deck', 'embark_town', 'passengerId', 'name',
       'age_imputed', 'fare_imputed', 'is_child', 'family_size', 'class',
       'embarked', 'fare_bin', 'age_fare_ratio', 'age_group', 'surname'],
      dtype='object')


**You also could just drop the artificial index column...** `df.drop(columns=['col'], inplace=True)`

### Q 4.3 Export the dataset to `csv`. Set `index=False` as an additional param.

In [41]:
df.to_csv('outputs/titanic_clean.csv', index=False)

## 5. Final Clean Dataset

### Load the Clean Dataset

- **Test first with a different variable name to avoid overwriting by accident.**
Load `outputs/titanic_clean.csv` into a new variable. Verify row count, identifiers, column names, numeric values and the absence of an artificial index column. Confirm `raw` still matches the source file. The undefined zero-fare ratio may remain missing by design.

In [42]:
df_final = pd.read_csv('outputs/titanic_clean.csv')
assert df_final.shape == df.shape
assert df_final.columns.tolist() == df.columns.tolist()
assert df_final['passengerId'].is_unique
assert df_final['passengerId'].tolist() == df['passengerId'].tolist()
assert not df_final.columns.str.startswith('Unnamed:').any()
assert np.allclose(df_final[['age', 'fare']], df[['age', 'fare']])
pd.testing.assert_frame_equal(raw, pd.read_csv(data_path, sep=';'))
print('Validated export:', df_final.shape)

Validated export: (891, 22)
